# ML Coding Practice - 1

Use this file for practicing ML coding. We may have additional files as these grow long. Have headers for each concept and implementation to keep it accessible in the future.

In [3]:
# Common imports
import torch
import torch.nn as nn
import torch.nn.functional as F

## Multihead Self-Attention (MHA)

One of the most important pieces for the transformer model

In [14]:
# Write the canoncial class for this.
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads, max_seq_len = 512):
        super().__init__()

        assert d_model % num_heads == 0
        self.head_dim = d_model // num_heads
        self.num_heads = num_heads

        # Define the layers.
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        # Out proj.
        self.W_out = nn.Linear(d_model, d_model, bias=False)

        # Precompute the mask, register_buffer so it moves with .to(device).
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("mask", mask)

    def forward(self, x):
        # Batch, seq_len and dims (channels).
        B, T, C = x.shape

        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        # Break it up into multiple heads.
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)      # (B, nh, T, hd)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)      # (B, nh, T, hd)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)      # (B, nh, T, hd)

        # Compute scaled-dot product attention.
        att = q @ k.transpose(-2,-1) * (self.head_dim**-0.5)      # (B, nh, T, T)
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        # Combine.
        scores = att @ v      # (B, nh, T, hd)
        scores = scores.transpose(1, 2).contiguous().view(B, T, C)

        out = self.W_out(scores)
        return out


In [15]:
# --- test ---
B, T, C, n_heads = 2, 8, 64, 4
x = torch.randn(B, T, C)
mha = MultiHeadAttention(d_model=C, num_heads=n_heads, max_seq_len=T)

out = mha(x)
print(out.shape)

torch.Size([2, 8, 64])


In [16]:
x[1,1]

tensor([ 9.2535e-01, -9.3458e-01, -1.6440e-01,  1.5903e-01, -5.7435e-01,
        -3.2664e-02,  1.1358e+00,  8.5657e-01, -1.2135e+00, -7.5613e-02,
        -1.0249e-01, -1.8307e-01, -1.7937e+00, -1.3428e+00, -1.9259e-01,
        -6.6252e-02,  1.1320e+00, -7.4184e-01,  2.1568e-02, -1.7669e-03,
        -1.7552e-01, -5.2361e-01,  2.0473e-01, -4.4804e-01,  1.2041e+00,
        -3.7157e-01, -8.7688e-01, -7.1908e-01, -1.0752e+00, -2.6802e-01,
         1.7402e+00,  5.9872e-01,  1.0789e+00, -6.6952e-01,  2.5559e-01,
         1.3139e+00,  1.6465e+00, -7.7611e-01,  4.8159e-02, -5.5650e-01,
        -7.9127e-01, -2.1960e+00,  6.1267e-02, -1.0544e+00,  4.8984e-01,
         7.2099e-01, -1.7808e-01, -6.1071e-01, -1.4340e+00,  8.6040e-01,
         1.7835e+00, -1.0374e+00,  8.4181e-01,  2.9719e-01,  1.3131e+00,
         1.4437e+00, -2.8265e-01,  9.1696e-01, -2.5368e+00, -1.4426e+00,
        -1.3453e+00,  4.0347e-01,  5.3770e-02, -7.7269e-01])

In [18]:
# --- Test against nn.MultiheadAttention ---
torch.manual_seed(21)
B, T, C, n_heads = 2, 8, 64, 4
x = torch.randn(B, T, C)

mine = MultiHeadAttention(d_model=C, num_heads=n_heads, max_seq_len=T)

ref = nn.MultiheadAttention(embed_dim=C, num_heads=n_heads, bias=False, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mine.W_q.weight, mine.W_k.weight, mine.W_v.weight], dim=0))
    ref.out_proj.weight.copy_(mine.W_out.weight)

causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # True = masked out

out_mine = mine(x)
out_ref, _ = ref(x, x, x, attn_mask=causal_mask, need_weights=False)

print(torch.allclose(out_mine, out_ref, atol=1e-5))
print((out_mine - out_ref).abs().max())

True
tensor(1.1921e-07, grad_fn=<MaxBackward1>)


## Transformer block

We have created this in `./karpathy_content/gpt_from_scratch/gpt.py` but doing it here with the pytorch MHA implementation as it could be useful to learn.

In [22]:
class TransformerBlock(nn.Module):
    """Transformer block: communication followed by computation, using native PyTorch modules."""

    def __init__(self, n_embd, n_heads, max_seq_len, dropout=0.0):
        super().__init__()

        self.mha = nn.MultiheadAttention(embed_dim=n_embd,
                                         num_heads=n_heads,
                                         dropout=dropout,
                                         bias=False,
                                         batch_first=True)
        
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd, bias=False),
            nn.Dropout(dropout)
            )
        self.ln1 = nn.RMSNorm(n_embd)
        self.ln2 = nn.RMSNorm(n_embd)

        # Precompute the causal mask.
        mask = torch.triu(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        x_norm = self.ln1(x)
        attn_out, _ = self.mha(x_norm, x_norm, x_norm, attn_mask = self.mask[:T, :T], need_weights=False)
        
        x = x + attn_out
        x = x + self.ff(self.ln2(x))

        return x

In [23]:
# --- test ---
torch.manual_seed(0)
B, T, C, n_heads = 2, 8, 64, 4
x = torch.randn(B, T, C)

block = TransformerBlock(n_embd=C, n_heads=n_heads, max_seq_len=T)
out = block(x)
print(out.shape)                                   # should be torch.Size([2, 8, 64])
print(sum(p.numel() for p in block.parameters()))   # sanity check that params are registered

torch.Size([2, 8, 64])
49280


## LayerNorm and RMSNorm

In [28]:
class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d))
        self.beta = nn.Parameter(torch.zeros(d))

    def forward(self, x):
        # x has a shape like (B, T, d) or more generally (..., d)
        x_mean = x.mean(dim=-1, keepdim=True)                    # (..., 1)
        # NOTE: biased variance (divide by d, not d-1) -- this is what nn.LayerNorm uses.
        x_var = x.var(dim=-1, keepdim=True, unbiased=False)      # (..., 1)
        x_norm = (x - x_mean) / torch.sqrt(x_var + self.eps)
        return self.gamma * x_norm + self.beta

In [29]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d))

    def forward(self, x):
        # x has a shape like (B, T, d) or more generally (..., d)
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x / rms)

In [30]:
# --- Test LayerNorm / RMSNorm against PyTorch ---
torch.manual_seed(21)
B, T, d = 2, 8, 64
x = torch.randn(B, T, d) * 3.0 + 1.5   # non-unit mean/std so the norm actually does work


def check(name, mine, ref, x, atol=1e-5):
    om, orf = mine(x), ref(x)
    print(f"{name:11s} shape={tuple(om.shape)}  allclose={torch.allclose(om, orf, atol=atol)}  "
          f"max_diff={(om - orf).abs().max().item():.2e}")


# Default affine params (gamma=1, beta=0). eps must match the reference.
check("LayerNorm", LayerNorm(d, eps=1e-5), nn.LayerNorm(d, eps=1e-5), x)
# nn.RMSNorm's default eps is dtype-dependent, so pass ours explicitly.
check("RMSNorm", RMSNorm(d, eps=1e-6), nn.RMSNorm(d, eps=1e-6), x)

# Again with non-trivial gamma/beta, to confirm the affine params are applied correctly.
ln_mine, ln_ref = LayerNorm(d, eps=1e-5), nn.LayerNorm(d, eps=1e-5)
rms_mine, rms_ref = RMSNorm(d, eps=1e-6), nn.RMSNorm(d, eps=1e-6)
with torch.no_grad():
    g, b = torch.randn(d), torch.randn(d)
    ln_mine.gamma.copy_(g); ln_mine.beta.copy_(b)
    ln_ref.weight.copy_(g);  ln_ref.bias.copy_(b)
    rms_mine.gamma.copy_(g); rms_ref.weight.copy_(g)

check("LayerNorm*", ln_mine, ln_ref, x)
check("RMSNorm*", rms_mine, rms_ref, x)

# Sanity properties: LN output is zero-mean / unit-var, RMS output has unit RMS (gamma=1).
y = LayerNorm(d)(x)
print(f"LN  mean~0: {y.mean(-1).abs().max():.2e}   std~1: {y.std(-1, unbiased=False).mean():.4f}")
z = RMSNorm(d)(x)
print(f"RMS rms~1: {z.pow(2).mean(-1).sqrt().mean():.4f}")

# Grads flow into the learnable params.
ln = LayerNorm(d); ln(x).sum().backward()
rn = RMSNorm(d);  rn(x).sum().backward()
print("grads:", ln.gamma.grad is not None, ln.beta.grad is not None, rn.gamma.grad is not None)

LayerNorm   shape=(2, 8, 64)  allclose=True  max_diff=3.58e-07
RMSNorm     shape=(2, 8, 64)  allclose=True  max_diff=2.38e-07
LayerNorm*  shape=(2, 8, 64)  allclose=True  max_diff=7.15e-07
RMSNorm*    shape=(2, 8, 64)  allclose=True  max_diff=4.77e-07
LN  mean~0: 6.71e-08   std~1: 1.0000
RMS rms~1: 1.0000
grads: True True True


## Softmax function

In [4]:
def softmax(x):
    # x can be a tensor of any shape. We will do the operations on the last dim.
    exp_x = torch.exp(x - x.max(dim=-1, keepdims=True).values)
    return exp_x / torch.sum(exp_x, axis=-1, keepdim=True)

In [5]:
# --- Test softmax against F.softmax ---
torch.manual_seed(21)

def check_softmax(name, x, atol=1e-6):
    mine, ref = softmax(x), F.softmax(x, dim=-1)
    row_sums = mine.sum(dim=-1)
    print(f"{name:14s} rows_sum_to_1={torch.allclose(row_sums, torch.ones_like(row_sums), atol=atol)}  "
          f"allclose={torch.allclose(mine, ref, atol=atol)}  "
          f"max_diff={(mine - ref).abs().max().item():.2e}")

check_softmax("2D", torch.randn(4, 10))
check_softmax("3D (B,T,T)", torch.randn(2, 8, 8))
check_softmax("large logits", torch.randn(4, 10) * 100)     # would overflow without the max-subtraction
check_softmax("with -inf", torch.randn(4, 10).masked_fill(torch.triu(torch.ones(4, 10, dtype=torch.bool), diagonal=1), float("-inf")))

# Non-negative and each row is a valid distribution.
p = softmax(torch.randn(4, 10))
print("all >= 0:", (p >= 0).all().item())

# Shift invariance: softmax(x) == softmax(x + c).
x = torch.randn(4, 10)
print("shift invariant:", torch.allclose(softmax(x), softmax(x + 5.0), atol=1e-6))


2D             rows_sum_to_1=True  allclose=True  max_diff=2.98e-08
3D (B,T,T)     rows_sum_to_1=True  allclose=True  max_diff=5.96e-08
large logits   rows_sum_to_1=True  allclose=True  max_diff=2.98e-08
with -inf      rows_sum_to_1=True  allclose=True  max_diff=2.98e-08
all >= 0: True
shift invariant: True
